In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Learing a neural barrier certificate for Ribosome Flow Model (500D) example.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches
import math

In [ ]:
def calc_h_rfm(lambdas, safety=0.8):

    lambdas = np.asarray(lambdas)
    pair_sums = lambdas[:-1] + lambdas[1:]
    alpha = np.max(pair_sums)

    if alpha <= 0:
        raise ValueError("lambda values must be positive.")
    if not (0 < safety < 1):
        raise ValueError("safety must be between 0 and 1.")

    return safety / alpha


def truncate_leading(hh):
    if hh == 0:
        return 0.0
    exp = math.floor(math.log10(abs(hh)))
    scale = 10 ** exp
    return math.floor(hh / scale) * scale


def simulate_rfm_discrete(n, lambdas, h, x0, steps=200):

    x = np.asarray(x0, dtype=float).copy()
    if x.shape != (n,):
        raise ValueError(f"x0 must have shape ({n},), got {x.shape}")

    lambdas = np.asarray(lambdas, dtype=float)
    if lambdas.shape != (n + 1,):
        raise ValueError(f"lambdas must have shape ({n+1},)")

    X = np.zeros((steps + 1, n))
    X[0] = x

    for t in range(steps):
        x_next = np.zeros_like(x)

        
        x_next[0] = x[0] + h * (
            lambdas[0] * (1 - x[0])
            - lambdas[1] * x[0] * (1 - x[1])
        )

        
        for i in range(1, n - 1):
            x_next[i] = x[i] + h * (
                lambdas[i] * x[i - 1] * (1 - x[i])
                - lambdas[i + 1] * x[i] * (1 - x[i + 1])
            )

        
        x_next[n - 1] = x[n - 1] + h * (
            lambdas[n - 1] * x[n - 2] * (1 - x[n - 1])
            - lambdas[n] * x[n - 1]
        )

        
        x_next = np.clip(x_next, 0.0, 1.0)

        x = x_next
        X[t + 1] = x

    return X

In [ ]:
# System Definition

t1 = time.perf_counter()

n = 500

rng = np.random.default_rng(0)
lambdas_np = rng.uniform(0.8, 1.2, size=n + 1).astype(np.float32)
lambdas = tf.constant(lambdas_np, dtype=tf.float32)

hh = calc_h_rfm(lambdas, safety=0.99)
h = truncate_leading(hh)

def f_rfm(x):

    x = tf.convert_to_tensor(x, dtype=tf.float32)

    squeeze = (x.shape.rank == 1)
    if squeeze:
        x = x[None, :]

    batch_size = tf.shape(x)[0]
    n_tf = tf.shape(x)[1]

    x_next = tf.TensorArray(tf.float32, size=n_tf)

    x1_next = x[:, 0] + h * (
        lambdas[0] * (1.0 - x[:, 0])
        - lambdas[1] * x[:, 0] * (1.0 - x[:, 1])
    )
    x_next = x_next.write(0, x1_next)

    
    for i in tf.range(1, n_tf - 1):
        xi_next = x[:, i] + h * (
            lambdas[i] * x[:, i - 1] * (1.0 - x[:, i])
            - lambdas[i + 1] * x[:, i] * (1.0 - x[:, i + 1])
        )
        x_next = x_next.write(i, xi_next)

    
    xn_next = x[:, -1] + h * (
        lambdas[-2] * x[:, -2] * (1.0 - x[:, -1])
        - lambdas[-1] * x[:, -1]
    )
    x_next = x_next.write(n_tf - 1, xn_next)

    x_next = tf.transpose(x_next.stack())  

    x_next = tf.clip_by_value(x_next, 0.0, 1.0)

    if squeeze:
        x_next = x_next[0]

    return x_next


In [ ]:
# Sampled points for a coarse gridding


xin_bar = np.ones((1,n))*0.2
xin_und = np.zeros((1,n))
xuns_und = np.eye(n)*0.9
xuns_bar = np.ones((1,n))*1
xi_bar= np.ones((1,n))*0.9
xi_und= np.zeros((1,n))

In [ ]:
# System Inductive evaluation for K=2500

def f_rfm_0_km1(x0, k):

    x = tf.convert_to_tensor(x0, dtype=tf.float32)

    if x.shape.rank == 1:
        x = x[None, :]

    batch_size = tf.shape(x)[0]
    n_tf = tf.shape(x)[1]

    traj = tf.TensorArray(tf.float32, size=k)

    for t in tf.range(k):
        traj = traj.write(t, x)
        x = f_rfm(x)

    return traj.stack()


def f_rfm_k(x0, k):

    x = tf.convert_to_tensor(x0, dtype=tf.float32)

    squeeze = (x.shape.rank == 1)
    if squeeze:
        x = x[None, :]

    for _ in tf.range(k):
        x = f_rfm(x)

    return x[0] if squeeze else x

k=2500

f_rfm_0km1_xin_bar = tf.squeeze(f_rfm_0_km1(xin_bar,k))
f_rfm_0km1_xi_und = tf.squeeze(f_rfm_0_km1(xi_und,k), axis=1)
f_rfm_k_xi_bar = f_rfm_k(xi_bar,k)

In [ ]:
# Neural Barrier Definition


class ScalarMonotoneNN(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        if activation is None:
            activation = tf.keras.layers.LeakyReLU(alpha=0.1)

        self.hidden_layers = []
        for h in hidden_dims:
            self.hidden_layers.append(
                layers.Dense(
                    h,
                    activation=activation,
                    use_bias=True,
                    kernel_constraint=constraints.NonNeg()
                )
            )

        self.out_layer = layers.Dense(
            1,
            activation=output_activation,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

    def call(self, x):
        z = x
        for layer in self.hidden_layers:
            z = layer(z)
        return self.out_layer(z)


class MaxAffineBranch(layers.Layer):

    def __init__(self, m_i, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.affine = layers.Dense(
            m_i,
            activation=None,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        z = self.affine(x)                           
        g = tf.reduce_max(z, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                        
        return y


class InputMaxBranch(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        g = tf.reduce_max(x, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                       
        return y


class ThreeBranchMinMonotoneNetwork(Model):

    def __init__(self,
                 input_dim,
                 m1,
                 m2,
                 hidden_dims1=(200, 200),
                 hidden_dims2=(200, 200),
                 hidden_dims3=(200, 200),
                 activation=None,
                 output_activation='linear'):
        super().__init__()

        self.input_dim = input_dim


        self.branch1 = InputMaxBranch(
            hidden_dims=hidden_dims1,
            activation=activation,
            output_activation=output_activation
        )


        self.branch2 = MaxAffineBranch(
            m_i=m1,
            hidden_dims=hidden_dims2,
            activation=activation,
            output_activation=output_activation
        )


        self.branch3 = MaxAffineBranch(
            m_i=m2,
            hidden_dims=hidden_dims3,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        y1 = self.branch1(x)   
        y2 = self.branch2(x)  
        y3 = self.branch3(x)   

        Y = tf.concat([y1, y2, y3], axis=1)          
        y = tf.reduce_min(Y, axis=1, keepdims=True)  
        return y


modelB = ThreeBranchMinMonotoneNetwork(
    input_dim=n,
    m1=int(0.1*n),   
    m2=int(0.2*n),   
    hidden_dims1=(200, 200),     
    hidden_dims2=(200, 200),
    hidden_dims3=(200, 200),
    activation=tf.keras.layers.LeakyReLU(alpha=0.1),
    output_activation='linear'
)

In [ ]:
# Training Monotone Neural Inductive Barrier

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)

gamma = -5
eta = 5


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.square(tf.nn.relu(tf.reduce_max(modelB(f_rfm_0km1_xin_bar))- gamma))
        loss_B += tf.square(tf.nn.relu(eta-tf.reduce_min(modelB(xuns_und))))
        loss_B += tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_rfm_0km1_xi_und)) - (eta-0.1))))  * (tf.nn.relu(modelB(f_rfm_k_xi_bar) -  (eta-0.1)))

        third_loss_B = tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_rfm_0km1_xi_und)) - (eta-0.1))))  * (tf.nn.relu(modelB(f_rfm_k_xi_bar) -  (eta-0.1)))

        third_loss_B_val = third_loss_B.numpy().item()

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
                
        if loss_B == 0   and i>1:
            print('Safety Verified')
            break
            
        if loss_B!=0:
            opt1.apply_gradients(zip(grad_B, modelB.trainable_variables),)
            
        if i%5 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_rfm_0km1_xin_bar)).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B.numpy().item():.4f}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_rfm_0km1_xin_bar)).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B.numpy().item():.4f}"
            )